# Stage 3 — Catheter / guidewire tracking (Colab/Kaggle BUILD)

**Thin notebook** — logic in `src/*`. YOLO11n (2 classes: catheter, guidewire) on **CathAction**, then **ByteTrack** on top for continuity. ByteTrack is Python around the detector, not inside the CoreML graph.

Acceptance is **fps + ID-switches**, not just IoU. CathAction is ~500k frames — on Kaggle attach it as a Dataset and selectively use a subset; don't copy all of it to Drive.

In [ ]:
import os, sys
REPO = '/content/drive/MyDrive/intv-img/interventional-imaging-pipeline'
if not os.path.exists(REPO):
    !git clone <YOUR_REPO_URL> {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install ultralytics>=8.2 pycocotools opencv-python pyyaml
from src import env
E = env.setup()
assert E['device'] == 'cuda', 'Switch runtime to GPU.'

In [ ]:
# 1) CathAction -> YOLO (catheter, guidewire). Importable converter (COCO or mask fallback).
import yaml
from src.data_prep import cathaction_to_yolo
cfg = yaml.safe_load(open('configs/catheter_track.yaml'))
cathaction_to_yolo.main(cfg)

In [ ]:
# 2) Train the 2-class detector on GPU (reuses the generalized train_detector).
from src.train.train_detector import train
best = train(cfg, project=f"{E['runs']}/catheter")
print('best ->', best)

In [ ]:
# 3) Track with ByteTrack on a clip; reports fps + unique track IDs (proxy for ID-switch behaviour).
from src.serve.track import track_yolo
track_yolo(best, source='data/raw/cathaction/sample_clip.mp4', out=f"{E['runs']}/catheter/track.mp4")

## Handoff
Copy `best.pt` to the Mac: `make export-coreml-yolo MODEL=best.pt` → `.mlpackage`, then run per-frame edge detection with `src.serve.realtime --task det`; ByteTrack stays in the app loop (`src.serve.track`).